## Part 2
This part focuses on using `spark` to analyze NFL data.

The code below imports all required modules for this entire section and runs objects needed to suppress/manage unnecessary warnings.

In [1]:
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

import pandas as pd
import numpy as np
import pyspark.pandas as ps

ps.set_option('compute.fail_on_ansi_mode', False)

from pyspark.sql import SparkSession
from pyspark.sql.functions import *

#code needed to manage package warnings -- this spark object is used for both pandas-on-spark and Spark SQL
spark = SparkSession.builder.appName('my_app').config('spark.sql.ansi.enabled', 'false').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 15:46:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/22 15:46:02 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


#### `pandas`-on-Spark

The code below reads in the weekly NFL data from the main filepath in JupyterHub.

In [2]:
NFL = ps.read_csv("weekly_nfl_data.csv")

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


Next, we use `.head()` to check out the first 5 rows of the `NFL` dataframe.

In [3]:
NFL.head()

26/03/22 15:46:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


The code below reports all the column names of the `NFL` dataframe.

In [4]:
NFL.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

Next, we only want to look at QB stats for the seasons 2005 to 2023 (inclusive). Additionally:
- We only want the following columns -
    - `player_display_name`
    - `season`
    - `week`
    - `completions`
    - `attempts`
    - `passing_yards`
    - `passing_tds`
    - `interceptions`
- For each combination of `player_display_name` and `season`, we will find the *sum* and *mean* of each of the statistical quantities in this subsetted dataset
- We will create two new variables by player/season combination:
    - `completion_percentage` = (sum of completions) / (sum of attempts)
    - `td_int_ratio` = (sum of passing tds) / (sum of interceptions)

The code below subsets `NFL` to look at regular season QB stats, specifically for seasons 2005 to 2023, inclusive. The code also subsets the columns of interest. This new object is called `NFL_sub1`.

In [6]:
NFL_sub1 = NFL.loc[(NFL['position'] == 'QB') & 
                   (NFL['season_type'] == 'REG') & 
                   (NFL['season'].between(2005, 2023))] \
              [["player_display_name", "season", "week", "completions",
                "attempts", "passing_yards", "passing_tds", "interceptions"]]

Next, the code below calculates the sum and mean for each of the statistical quantities in this subsetted data for each combination of `player_display_name` and `season`. This new object is called `NFL_sub2`.

In [7]:
NFL_sub2 = NFL_sub1.groupby(["player_display_name", "season"]) \
                    [["completions", "attempts", "passing_yards",
                      "passing_tds", "interceptions"]].agg(['mean', 'sum'])

Finally, we create the two new variables that were previously described! Then, they are added to our already subsetted dataframe (`NFL_sub2`).

In [8]:
cp = NFL_sub2["completions", "sum"] / NFL_sub2["attempts", "sum"] #creates completion_percentage variable
NFL_sub2["completion_percentage"] = cp #adds completion_percentage variable to NFL_sub2

tdir = NFL_sub2["passing_tds", "sum"] / NFL_sub2["interceptions", "sum"] #creates td_int_ratio variable
NFL_sub2["td_int_ratio"] = tdir #adds td_int_ratio variable to NFL_sub2

NFL_sub2.head() #previews final df

completions        attempts      passing_yards         passing_tds     interceptions       completion_percentage td_int_ratio
                                  mean  sum       mean  sum          mean     sum        mean sum          mean   sum                                   
player_display_name season                                                                                                                              
Jake Delhomme       2006     20.230769  263  33.153846  431    215.769231  2805.0    1.307692  17      0.846154  11.0              0.610209     1.545455
Jake Plummer        2005     17.312500  277  28.500000  456    210.375000  3366.0    1.125000  18      0.437500   7.0              0.607456     2.571429
Matt Schaub         2006      3.600000   18   5.400000   27     41.600000   208.0    0.200000   1      0.400000   2.0              0.666667     0.500000
Vince Young         2006     12.266667  184  23.733333  356    146.600000  2199.0    0.800000  12      0.866667  13.0              0.516854     0.923077
Kerry Collins       2007      8.333333   50  13.666667   82     88.500000   531.0    0.000000   0      0.000000   0.0              0.609756          NaN

The final results of the above are saved in the `NFL_sub2` object.

Now, we take `NFL_sub2` and do the following:
- Subset the rows to only include player/season combinations where the sum of attempts is at least 50
- Sort the rows descending by `completion_percentage` and report the first 40 values
- Sort the rows descending by `td_int_ratio` and report the first 40 values

First, we subset the `NFL_sub2` object to only include the rows where the player/season combination sum of attempts is at least 50. This new object is called `NFL_sub3`.

In [9]:
NFL_sub3 = NFL_sub2.loc[NFL_sub2["attempts", "sum"] >= 50]

Next, we sort the rows of `NFL_sub3` descending by `completion_percentage` and report the first 40 values.

In [10]:
NFL_sub3.sort_values("completion_percentage", ascending=False).head(40)

completions        attempts      passing_yards         passing_tds     interceptions       completion_percentage td_int_ratio
                                  mean  sum       mean  sum          mean     sum        mean sum          mean   sum                                   
player_display_name season                                                                                                                              
C.J. Beathard       2023      6.666667   40   8.833333   53     58.166667   349.0    0.166667   1      0.000000   0.0              0.754717          inf
Colt McCoy          2021     10.571429   74  14.142857   99    105.714286   740.0    0.428571   3      0.142857   1.0              0.747475     3.000000
Matt Schaub         2019     10.000000   50  13.400000   67    116.000000   580.0    0.600000   3      0.200000   1.0              0.746269     3.000000
Drew Brees          2018     24.266667  364  32.600000  489    266.133333  3992.0    2.133333  32      0.333333   5.0              0.744376     6.400000
                    2019     25.545455  281  34.363636  378    270.818182  2979.0    2.454545  27      0.363636   4.0              0.743386     6.750000
Mason Rudolph       2023     13.750000   55  18.500000   74    179.750000   719.0    0.750000   3      0.000000   0.0              0.743243          inf
Taysom Hill         2020      5.500000   88   7.562500  121     58.000000   928.0    0.250000   4      0.125000   2.0              0.727273     2.000000
Nick Foles          2018     28.200000  141  39.000000  195    282.600000  1413.0    1.400000   7      0.800000   4.0              0.723077     1.750000
Drew Brees          2017     24.125000  386  33.500000  536    270.875000  4334.0    1.437500  23      0.500000   8.0              0.720149     2.875000
Sam Bradford        2016     26.333333  395  36.800000  552    258.466667  3877.0    1.333333  20      0.333333   5.0              0.715580     4.000000
Drew Brees          2011     29.437500  471  41.250000  660    345.937500  5535.0    2.875000  46      0.875000  14.0              0.713636     3.285714
Colt McCoy          2014     18.200000   91  25.600000  128    211.400000  1057.0    0.800000   4      0.600000   3.0              0.710938     1.333333
Aaron Rodgers       2020     23.250000  372  32.875000  526    268.687500  4299.0    3.000000  48      0.312500   5.0              0.707224     9.600000
Bailey Zappe        2022     16.250000   65  23.000000   92    195.250000   781.0    1.250000   5      0.750000   3.0              0.706522     1.666667
Drew Brees          2009     24.200000  363  34.266667  514    292.533333  4388.0    2.266667  34      0.733333  11.0              0.706226     3.090909
                    2020     22.916667  275  32.500000  390    245.166667  2942.0    2.000000  24      0.500000   6.0              0.705128     4.000000
Joe Burrow          2021     22.875000  366  32.500000  520    288.187500  4611.0    2.125000  34      0.875000  14.0              0.703846     2.428571
Derek Carr          2019     22.562500  361  32.062500  513    253.375000  4054.0    1.312500  21      0.500000   8.0              0.703704     2.625000
Jake Browning       2023     19.000000  171  27.000000  243    215.111111  1936.0    1.333333  12      0.777778   7.0              0.703704     1.714286
Chase Daniel        2019     15.000000   45  21.333333   64    145.000000   435.0    1.000000   3      0.666667   2.0              0.703125     1.500000
Ryan Tannehill      2019     16.750000  201  23.833333  286    228.500000  2742.0    1.833333  22      0.500000   6.0              0.702797     3.666667
Deshaun Watson      2020     23.875000  382  34.000000  544    301.437500  4823.0    2.062500  33      0.437500   7.0              0.702206     4.714286
Alex Smith          2012     15.300000  153  21.800000  218    173.700000  1737.0    1.300000  13      0.500000   5.0              0.701835     2.600000
Kirk Cousins        2018     26.562500  425  37.8

Finally, independent of the previous step, we again sort the rows of `NFL_sub3`, this time descending by `td_int_ratio` and report the first 40 values.

In [11]:
NFL_sub3.sort_values("td_int_ratio", ascending=False).head(40)

completions        attempts      passing_yards         passing_tds     interceptions       completion_percentage td_int_ratio
                                  mean  sum       mean  sum          mean     sum        mean sum          mean   sum                                   
player_display_name season                                                                                                                              
Charlie Batch       2006      4.285714   30   7.428571   52     68.142857   477.0    0.714286   5      0.000000   0.0              0.576923          inf
Matt Schaub         2005      4.714286   33   9.142857   64     70.714286   495.0    0.571429   4      0.000000   0.0              0.515625          inf
Todd Collins        2007     16.750000   67  26.250000  105    222.000000   888.0    1.250000   5      0.000000   0.0              0.638095          inf
Troy Smith          2007     10.000000   40  19.000000   76    113.000000   452.0    0.500000   2      0.000000   0.0              0.526316          inf
Jake Locker         2011      6.800000   34  13.200000   66    108.400000   542.0    0.800000   4      0.000000   0.0              0.515152          inf
Brian Hoyer         2016     22.333333  134  33.333333  200    240.833333  1445.0    1.000000   6      0.000000   0.0              0.670000          inf
Nick Foles          2016     18.000000   36  27.500000   55    205.000000   410.0    1.500000   3      0.000000   0.0              0.654545          inf
Derek Anderson      2014     13.000000   65  19.400000   97    140.200000   701.0    1.000000   5      0.000000   0.0              0.670103          inf
Jimmy Garoppolo     2016      7.166667   43  10.500000   63     83.666667   502.0    0.666667   4      0.000000   0.0              0.682540          inf
Matt Moore          2019      9.833333   59  15.166667   91    109.833333   659.0    0.666667   4      0.000000   0.0              0.648352          inf
C.J. Beathard       2020     11.000000   66  17.333333  104    131.166667   787.0    1.000000   6      0.000000   0.0              0.634615          inf
Andy Dalton         2023     17.000000   34  29.000000   58    180.500000   361.0    1.000000   2      0.000000   0.0              0.586207          inf
Mason Rudolph       2023     13.750000   55  18.500000   74    179.750000   719.0    0.750000   3      0.000000   0.0              0.743243          inf
Desmond Ridder      2022     18.250000   73  28.750000  115    177.000000   708.0    0.500000   2      0.000000   0.0              0.634783          inf
C.J. Beathard       2023      6.666667   40   8.833333   53     58.166667   349.0    0.166667   1      0.000000   0.0              0.754717          inf
Tom Brady           2016     24.250000  291  36.000000  432    296.166667  3554.0    2.333333  28      0.166667   2.0              0.673611    14.000000
Nick Foles          2013     15.615385  203  24.384615  317    222.384615  2891.0    2.076923  27      0.153846   2.0              0.640379    13.500000
Josh McCown         2013     18.625000  149  28.000000  224    228.625000  1829.0    1.625000  13      0.125000   1.0              0.665179    13.000000
Aaron Rodgers       2018     23.250000  372  37.312500  597    277.625000  4442.0    1.562500  25      0.125000   2.0              0.623116    12.500000
Damon Huard         2006     14.800000  148  24.400000  244    187.800000  1878.0    1.100000  11      0.100000   1.0              0.606557    11.000000
Aaron Rodgers       2020     23.250000  372  32.875000  526    268.687500  4299.0    3.000000  48      0.312500   5.0              0.707224     9.600000
                    2021     22.875000  366  33.187500  531    257.187500  4115.0    2.312500  37      0.250000   4.0              0.689266     9.250000
Tom Brady           2010     20.250000  324  30.750000  492    243.750000  3900.0    2.250000  36      0.250000   4.0              0.658537     9.000000
Jake Delhomme       2007     18.333333   55  28.6

#### Spark SQL

The code below reads in the weekly NFL data from the main filepath in JupyterHub.

In [12]:
NFL_spark = spark.read.load("weekly_nfl_data.csv",
                            format="csv",
                            sep=",",
                            inferSchema="true",
                            header="true")

Next, we use `.show(5)` to check out the first 5 rows of the `NFL_spark` dataframe.

In [13]:
NFL_spark.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

The code below reports all the column names of the `NFL_spark` dataframe.

In [14]:
NFL_spark.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

Next, we only want to look at QB stats for the seasons 2005 to 2023 (inclusive). Additionally:
- We only want the following columns -
    - `player_display_name`
    - `season`
    - `week`
    - `completions`
    - `attempts`
    - `passing_yards`
    - `passing_tds`
    - `interceptions`
- For each combination of `player_display_name` and `season`, we will find the *sum* and *mean* of each of the statistical quantities in this subsetted dataset
- We will create two new variables by player/season combination:
    - `completion_percentage` = (sum of completions) / (sum of attempts)
    - `td_int_ratio` = (sum of passing tds) / (sum of interceptions)

The code below subsets `NFL_spark` to meet all of the above criteria and also adds the two new variables. This is all accomplished in the one code chunk below. The results are saved as an object called `NFL_spark_ss`.

In [15]:
NFL_spark_ss = NFL_spark.filter((NFL_spark.position == 'QB') & (NFL_spark.season_type == 'REG') &
                                (NFL_spark.season.between(2005,2023))) \
                        .select(["player_display_name", "season", "week", "completions",
                                 "attempts", "passing_yards", "passing_tds", "interceptions"]) \
                        .groupby(["player_display_name", "season"]) \
                        .agg(sum("completions"), avg("completions"), sum("attempts"), avg("attempts"),
                             sum("passing_yards"), avg("passing_yards"), sum("passing_tds"), avg("passing_tds"),
                             sum("interceptions"), avg("interceptions")) \
                        .withColumn("completion_percentage", col("sum(completions)") / col("sum(attempts)")) \
                        .withColumn("td_int_ratio", col("sum(passing_tds)") / col("sum(interceptions)"))

NFL_spark_ss.show(5) #previews resulting df

+-------------------+------+----------------+------------------+-------------+------------------+------------------+------------------+----------------+------------------+------------------+------------------+---------------------+------------------+
|player_display_name|season|sum(completions)|  avg(completions)|sum(attempts)|     avg(attempts)|sum(passing_yards)|avg(passing_yards)|sum(passing_tds)|  avg(passing_tds)|sum(interceptions)|avg(interceptions)|completion_percentage|      td_int_ratio|
+-------------------+------+----------------+------------------+-------------+------------------+------------------+------------------+----------------+------------------+------------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|             263| 20.23076923076923|          431| 33.15384615384615|            2805.0|215.76923076923077|              17|1.3076923076923077|              11.0|0.8461538461538461|   0.6102088167053364|1.54545454545454

Now, we take `NFL_spark_ss` and do the following:
- Subset the rows to only include player/season combinations where the sum of attempts is at least 50.
- Sort the rows descending by completion_percentage and report the first 40 values!
- Sort the rows descending by td_int_ratio and report the first 40 values!

First, we show `NFL_spark_ss` subsetted to only include player/season combinations where the sum of attempts is at least 50. Then, we sort the rows descending by `completion_percentage` and report the first 40 values. This is all done in one code chunk.

In [16]:
NFL_spark_ss.filter(col("sum(attempts)") >= 50) \
            .sort(NFL_spark_ss.completion_percentage, ascending = False) \
            .show(40)

+-------------------+------+----------------+------------------+-------------+------------------+------------------+------------------+----------------+-------------------+------------------+-------------------+---------------------+------------------+
|player_display_name|season|sum(completions)|  avg(completions)|sum(attempts)|     avg(attempts)|sum(passing_yards)|avg(passing_yards)|sum(passing_tds)|   avg(passing_tds)|sum(interceptions)| avg(interceptions)|completion_percentage|      td_int_ratio|
+-------------------+------+----------------+------------------+-------------+------------------+------------------+------------------+----------------+-------------------+------------------+-------------------+---------------------+------------------+
|      C.J. Beathard|  2023|              40| 6.666666666666667|           53| 8.833333333333334|             349.0|58.166666666666664|               1|0.16666666666666666|               0.0|                0.0|   0.7547169811320755|        

Finally, we show `NFL_spark_ss` subsetted to only include player/season combinations where the sum of attempts is at least 50. Then, we sort the rows descending by `td_int_ratio` and report the first 40 values. This is all done in one code chunk.

In [17]:
NFL_spark_ss.filter(col("sum(attempts)") >= 50) \
            .sort(NFL_spark_ss.td_int_ratio, ascending = False) \
            .show(40)

+-------------------+------+----------------+------------------+-------------+------------------+------------------+------------------+----------------+------------------+------------------+-------------------+---------------------+-----------------+
|player_display_name|season|sum(completions)|  avg(completions)|sum(attempts)|     avg(attempts)|sum(passing_yards)|avg(passing_yards)|sum(passing_tds)|  avg(passing_tds)|sum(interceptions)| avg(interceptions)|completion_percentage|     td_int_ratio|
+-------------------+------+----------------+------------------+-------------+------------------+------------------+------------------+----------------+------------------+------------------+-------------------+---------------------+-----------------+
|          Tom Brady|  2016|             291|             24.25|          432|              36.0|            3554.0| 296.1666666666667|              28|2.3333333333333335|               2.0|0.16666666666666666|   0.6736111111111112|             14